# 流媒体
LangGraph 实施了流媒体系统来显示实时更新，从而实现响应迅速且透明的用户体验。

LangGraph 的流式传输系统可让您将图形运行的实时反馈显示到您的应用中。
您可以流式传输的数据主要有三类：

- 工作流进度——执行每个图形节点后获取状态更新。
- LLM 标记— 流式传输语言模型标记。
- 自定义更新——发出用户定义的信号（例如，“获取 10/100 条记录”）。

## LangGraph 流式传输有何潜力¶
- [Stream LLM tokens](https://langchain-ai.github.io/langgraph/how-tos/streaming/#messages)——从任何地方捕获令牌流：内部节点、子图或工具。
- [Emit progress notifications from tools](https://langchain-ai.github.io/langgraph/how-tos/streaming/#stream-custom-data)——直接从工具功能发送自定义更新或进度信号。
- [Stream from subgraphs](https://langchain-ai.github.io/langgraph/how-tos/streaming/#stream-subgraph-outputs)——包括来自父图和任何嵌套子图的输出。
- [Use any LLM](https://langchain-ai.github.io/langgraph/how-tos/streaming/#use-with-any-llm) — 从任何 LLM 流出令牌，即使它不是使用custom流模式的 LangChain 模型。
- [Use multiple streaming modes](https://langchain-ai.github.io/langgraph/how-tos/streaming/#stream-multiple-modes)- 从values（完整状态）、updates（状态增量）、messages（LLM 令牌 + 元数据）、custom（任意用户数据）或debug（详细跟踪）中选择

# 流输出
您可以从 LangGraph 代理或工作流中流式输出。

支持的流模式

将以下一个或多个流模式作为列表传递给`stream()`或`astream()`方法：

| 模式 | 描述 |
| :--- | :--- |
| values | 在图的每个步骤之后流式传输状态的完整值。 |
| updates | 将图的每个步骤之后的更新流式传输到状态。如果在同一步骤中进行了多个更新（例如，运行了多个节点），则这些更新将分别流式传输。 |
| custom | 从图形节点内部流式传输自定义数据。 |
| messages | 从调用 LLM 的任何图形节点流式传输 2 元组（LLM 令牌、元数据）。 |
| debug | 在整个图表执行过程中传输尽可能多的信息。 |



## 来自代理的流¶
### 代理进度
要流式传输代理进度，请将stream()或astream()方法与 结合使用stream_mode="updates"。这将在代理每个步骤后发出一个事件。

例如，如果您有一个调用工具一次的代理，您应该会看到以下更新：

- LLM 节点：带有工具调用请求的 AI 消息
- 工具节点：带有执行结果的工具消息
- LLM 节点：最终 AI 响应


In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
import os
load_dotenv()  # 加载环境变量

agent = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=[get_weather],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]},
    stream_mode="updates"
):
    print(chunk)
    print("\n")